# 🎬 Movie Recommendation System
### Content-Based Filtering using Cosine Similarity

---

**Approach:** Content-Based Filtering  
**Similarity Metric:** Cosine Similarity  
**Features Used:** Genre, Tags, Rating, Year  
**Libraries:** NumPy, Pandas, Scikit-learn, Matplotlib, Seaborn

---

## 📌 How It Works
1. Each movie is converted into a **feature vector** (genre flags + tag flags + normalized rating + year)
2. A **user profile vector** is built by averaging the vectors of liked movies
3. **Cosine similarity** is computed between the user profile and all other movies
4. Top-N most similar movies are returned as recommendations

## Step 1: Install & Import Libraries

In [ ]:
# All libraries are pre-installed in Google Colab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

print('✅ All libraries imported successfully!')

## Step 2: Create the Movie Dataset

In [ ]:
data = {
    'title':    ['Inception', 'The Dark Knight', 'Interstellar', 'The Matrix',
                 'Parasite', 'Spirited Away', 'The Godfather', 'Pulp Fiction',
                 'Avengers: Endgame', 'Your Name', 'Get Out',
                 'Mad Max: Fury Road', 'La La Land', 'Princess Mononoke', 'Blade Runner 2049'],

    'genre':    [['Sci-Fi','Thriller'], ['Action','Thriller'], ['Sci-Fi','Drama'],
                 ['Sci-Fi','Action'], ['Thriller','Drama'], ['Animation','Fantasy'],
                 ['Crime','Drama'], ['Crime','Drama'], ['Action','Sci-Fi'],
                 ['Animation','Romance'], ['Horror','Thriller'], ['Action','Sci-Fi'],
                 ['Romance','Drama'], ['Animation','Fantasy'], ['Sci-Fi','Drama']],

    'tags':     [['mind-bending','heist','dreams'], ['superhero','crime','justice'],
                 ['space','time','love'], ['virtual reality','dystopia','rebellion'],
                 ['class','social','dark comedy'], ['fantasy','spirit','coming-of-age'],
                 ['mafia','family','power'], ['nonlinear','crime','cult'],
                 ['superhero','time travel','epic'], ['romance','time','identity'],
                 ['race','social horror','twist'], ['post-apocalyptic','survival','chaos'],
                 ['music','dreams','nostalgia'], ['nature','war','fantasy'],
                 ['dystopia','identity','future']],

    'rating':   [8.8, 9.0, 8.6, 8.7, 8.6, 8.6, 9.2, 8.9, 8.4, 8.4, 7.7, 8.1, 8.0, 8.4, 8.0],
    'year':     [2010, 2008, 2014, 1999, 2019, 2001, 1972, 1994, 2019, 2016, 2017, 2015, 2016, 1997, 2017]
}

df = pd.DataFrame(data)
print(f'Dataset shape: {df.shape}')
df[['title','rating','year']].head(10)

## Step 3: Feature Engineering — Build Feature Vectors

We encode each movie into a numerical vector:
- **Genre** → One-hot encoding (e.g., `[1, 0, 1, 0, ...]`)
- **Tags** → One-hot encoding
- **Rating & Year** → Normalized to [0, 1] using MinMaxScaler

In [ ]:
# One-hot encode genres
mlb_genre = MultiLabelBinarizer()
genre_encoded = mlb_genre.fit_transform(df['genre'])
genre_df = pd.DataFrame(genre_encoded, columns=mlb_genre.classes_)

# One-hot encode tags
mlb_tags = MultiLabelBinarizer()
tags_encoded = mlb_tags.fit_transform(df['tags'])
tags_df = pd.DataFrame(tags_encoded, columns=mlb_tags.classes_)

# Normalize rating and year
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'year']])
num_df = pd.DataFrame(num_features, columns=['rating_norm', 'year_norm'])

# Combine all features
feature_matrix = pd.concat([genre_df, tags_df, num_df], axis=1)
feature_matrix.index = df['title']

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'  → {genre_df.shape[1]} genre features')
print(f'  → {tags_df.shape[1]} tag features')
print(f'  → 2 numerical features (rating, year)')
feature_matrix.head(3)

## Step 4: Compute Cosine Similarity Matrix

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

This gives a value between **0** (completely different) and **1** (identical).

In [ ]:
# Compute full similarity matrix
sim_matrix = cosine_similarity(feature_matrix)
sim_df = pd.DataFrame(sim_matrix, index=df['title'], columns=df['title'])

print(f'Similarity matrix shape: {sim_df.shape}')
print('\nSample similarities for "Inception":')
print(sim_df['Inception'].sort_values(ascending=False).head(6))

## Step 5: Visualize the Similarity Matrix

In [ ]:
plt.figure(figsize=(12, 9))
mask = np.eye(len(sim_df), dtype=bool)  # mask diagonal

sns.heatmap(
    sim_df, annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.4, linecolor='#e2e8f0',
    mask=mask, vmin=0, vmax=1,
    annot_kws={'size': 7}
)

plt.title('🎬 Movie Cosine Similarity Matrix', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()
print('Higher value = more similar movies')

## Step 6: Build the Recommendation Function

In [ ]:
def get_recommendations(liked_movies, df, feature_matrix, top_n=5):
    """
    Content-based recommendation using cosine similarity.

    Parameters:
        liked_movies (list): List of movie titles the user likes
        df (DataFrame): Movie dataset
        feature_matrix (DataFrame): Encoded feature vectors
        top_n (int): Number of recommendations to return

    Returns:
        DataFrame: Top-N recommended movies with similarity scores
    """
    # Validate input
    valid = [m for m in liked_movies if m in feature_matrix.index]
    if not valid:
        print('❌ None of the provided movies found in dataset.')
        return None

    # Step 1: Build user profile = average of liked movie vectors
    liked_vectors = feature_matrix.loc[valid].values
    user_profile = liked_vectors.mean(axis=0).reshape(1, -1)

    # Step 2: Compute cosine similarity between user profile and all movies
    scores = cosine_similarity(user_profile, feature_matrix)[0]

    # Step 3: Rank and filter out already-liked movies
    results = pd.DataFrame({
        'title': feature_matrix.index,
        'similarity_score': scores
    })
    results = results[~results['title'].isin(valid)]
    results = results.sort_values('similarity_score', ascending=False).head(top_n)

    # Step 4: Merge with movie metadata
    results = results.merge(df[['title','genre','rating','year']], on='title')
    results['match_%'] = (results['similarity_score'] * 100).round(1)
    results = results[['title','match_%','rating','year','genre']].reset_index(drop=True)
    results.index += 1  # start from 1

    return results

print('✅ Recommendation function defined!')

## Step 7: Get Recommendations — Try It Out!

In [ ]:
# --- 🎬 CHANGE THESE TO YOUR FAVOURITE MOVIES ---
liked = ['Inception', 'The Matrix']
# ------------------------------------------------

print(f'🎯 Finding recommendations based on: {liked}\n')
recs = get_recommendations(liked, df, feature_matrix, top_n=5)
print(recs.to_string())

## Step 8: Visualize Recommendations

In [ ]:
def plot_recommendations(recs, liked_movies):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(recs)))

    # --- Bar chart: match scores ---
    ax1 = axes[0]
    bars = ax1.barh(recs['title'][::-1], recs['match_%'][::-1], color=colors[::-1], edgecolor='white', height=0.6)
    ax1.set_xlabel('Match Score (%)', fontsize=11)
    ax1.set_title(f'Top Recommendations\n(based on: {", ".join(liked_movies)})', fontsize=11, fontweight='bold')
    ax1.set_xlim(0, 105)
    for bar, val in zip(bars, recs['match_%'][::-1]):
        ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val}%', va='center', fontsize=9, fontweight='bold', color='#334155')
    ax1.spines[['top','right']].set_visible(False)
    ax1.set_facecolor('#f8fafc')

    # --- Scatter: rating vs year ---
    ax2 = axes[1]
    all_other = df[~df['title'].isin(liked_movies) & ~df['title'].isin(recs['title'])]
    ax2.scatter(all_other['year'], all_other['rating'], color='#cbd5e1', s=60, label='Other movies', zorder=2)

    rec_data = df[df['title'].isin(recs['title'])]
    sc = ax2.scatter(rec_data['year'], rec_data['rating'],
                     c=recs.set_index('title').loc[rec_data['title'], 'match_%'].values,
                     cmap='Blues', s=150, zorder=3, edgecolors='#1e40af', linewidths=1)
    for _, row in rec_data.iterrows():
        ax2.annotate(row['title'], (row['year'], row['rating']),
                     textcoords='offset points', xytext=(5, 5), fontsize=7, color='#1e3a8a')

    liked_data = df[df['title'].isin(liked_movies)]
    ax2.scatter(liked_data['year'], liked_data['rating'],
                color='#f97316', s=180, zorder=4, marker='*', label='Liked movies')
    for _, row in liked_data.iterrows():
        ax2.annotate(row['title'], (row['year'], row['rating']),
                     textcoords='offset points', xytext=(5, -10), fontsize=7, color='#ea580c', fontweight='bold')

    plt.colorbar(sc, ax=ax2, label='Match %')
    ax2.set_xlabel('Year', fontsize=11)
    ax2.set_ylabel('IMDb Rating', fontsize=11)
    ax2.set_title('Recommendations: Rating vs Year', fontsize=11, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.spines[['top','right']].set_visible(False)
    ax2.set_facecolor('#f8fafc')

    plt.tight_layout()
    plt.show()

plot_recommendations(recs, liked)

## Step 9: Test Different Preferences

In [ ]:
test_cases = [
    ['The Godfather', 'Pulp Fiction'],
    ['Spirited Away', 'Your Name'],
    ['Avengers: Endgame', 'Mad Max: Fury Road'],
]

for liked_test in test_cases:
    print(f'\n{"="*55}')
    print(f'👤 User likes: {liked_test}')
    print('='*55)
    result = get_recommendations(liked_test, df, feature_matrix, top_n=3)
    print(result[['title','match_%','rating']].to_string())

## Step 10: Summary & Key Takeaways

| Concept | What We Did |
|---|---|
| **Feature Engineering** | One-hot encoded genres & tags; normalized rating & year |
| **User Profile** | Averaged feature vectors of liked movies |
| **Similarity Metric** | Cosine Similarity via `sklearn` |
| **Recommendation** | Ranked unwatched movies by similarity score |

### Pros & Cons of Content-Based Filtering
| ✅ Pros | ❌ Cons |
|---|---|
| No need for other users' data | Limited to features we define |
| Works for new users (just need preferences) | Misses surprises / serendipity |
| Transparent & explainable | Doesn't improve with more users |

### 🚀 Extensions to try
- Add **collaborative filtering** (SVD, KNN on user-item matrix)
- Use **TF-IDF** on movie descriptions instead of manual tags
- Build a **hybrid system** combining both approaches